# Predição do modelo de classificação de Coral-Sol utilizando o dataset anotado por Gabriela

Autor:  Matheus de Farias Cavalcanti Santos

Atualizado: 06/03/2026

## Objetivo:
Notebook destinado à predição das classes de Coral-Sol utilizando o dataset anotado por Gabriela.
- Classe POSITIVO: presença de Coral-Sol
- Classe NEGATIVO: ausência de Coral-Sol

🔗[Dataset anotado por Gabriela](https://petrobrasbr.sharepoint.com/teams/IASubmarina/Shared%20Documents/Forms/AllItems.aspx?id=%2Fteams%2FIASubmarina%2FShared%20Documents%2FIA%2DAnaliseAmbiental%2F9%20%2D%20Dataset%2DBinario%2DPPCEX%2DAgosto%2D2025%2FResultado&viewid=0ee9d069%2D96be%2D4cad%2D9f2d%2D9e53c19fe973&ct=1771932670845&or=Teams%2DHL&LOF=1&xsdata=MDV8MDJ8fDcwM2Q0NGFkNDAzNzQxYTg5OWZkMDhkZTczOTkxZTA3fDViNmY2MjQxOWE1NzRiZTQ4ZTUwMWRmYTcyZTc5YTU3fDB8MHw2MzkwNzUyOTg2MDQxOTg5MTh8VW5rbm93bnxWR1ZoYlhOVFpXTjFjbWwwZVZObGNuWnBZMlY4ZXlKRFFTSTZJbFJsWVcxelgwRlVVRk5sY25acFkyVmZVMUJQVEU5R0lpd2lWaUk2SWpBdU1DNHdNREF3SWl3aVVDSTZJbGRwYmpNeUlpd2lRVTRpT2lKUGRHaGxjaUlzSWxkVUlqb3hNWDA9fDF8TDJOb1lYUnpMekU1T2pNNE5tRTFaVEl3TFRnNE1ERXRORGN4TVMwNU16VTNMV1U0TURRd09EWXhOV1E0TVY5bU9XUXlOR0U0WWkxbE0yVTVMVFJtTXpNdE9HTTFOQzA0WXpnME4yUXdZV05rTkRsQWRXNXhMbWRpYkM1emNHRmpaWE12YldWemMyRm5aWE12TVRjM01Ua3pNekExT0RnMU13PT18ZGU0MjMwN2E2MDNiNGM4MmZkMmQwOGRlNzM5OTFlMDZ8ZjdiMmFlNDgwMTc5NDQ4ODlhMmJlMGNmZjEwYzFhNTY%3D&sdata=Ymc3bnp4MEZTRFVwWS9Rc0tpa1hEaHZsSlFvVWQ1cm1FRWMrbVoxSWhmZz0%3D&ovuser=5b6f6241-9a57-4be4-8e50-1dfa72e79a57%2Csuzane.lima.prestserv%40petrobras.com.br)

🔗[Link para o modelo de classificação](https://petrobrasbr.sharepoint.com/:f:/r/teams/IASubmarina/Shared%20Documents/Evid%C3%AAncias-OKRs/1%20-%20Arquivos%20de%20evid%C3%AAncias/2026/CICLO%201/OKR4/Modelo%20de%20classifica%C3%A7%C3%A3o%20de%20Coral%20-%20Sol?csf=1&web=1&e=MfcHxs)

🔗[Link para as imagens resultantes da execução desse notebook](https://petrobrasbr.sharepoint.com/:f:/r/teams/IASubmarina/Shared%20Documents/Evid%C3%AAncias-OKRs/1%20-%20Arquivos%20de%20evid%C3%AAncias/2026/CICLO%201/OKR4/Dataset%20Gabriela%20consolidado%2050%25?csf=1&web=1&e=ebwUbh)

## Importações Necessárias

In [ ]:
!pip install numpy opencv-python matplotlib ultralytics scikit-learn nbconvert torch tqdm

In [2]:
import numpy as np
import pandas as pd
import cv2
import glob
from pathlib import Path
import os
import matplotlib.pyplot as plt
from ultralytics import YOLO
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import sys

## Declaração de Constantes e Modelos

In [ ]:
MODEL_V1 = YOLO("./coral_yolov11_class_8c_V4.pt")

## Funções Necessárias

In [ ]:
def read_image(filename: str) -> np.ndarray:
    """
    Carrega uma imagem do disco, converte para o formato RGB e realiza um
    recorte vertical centralizado preservando a largura original.

    O recorte mantém aproximadamente 68% da altura da imagem (34% acima e
    34% abaixo do centro vertical).

    Args:
        filename (str): Caminho para o arquivo de imagem.

    Returns:
        np.ndarray: Imagem RGB recortada como um array uint8.
        None: Caso a imagem não possa ser carregada ou ocorra um erro.
    """
    try:
        image = cv2.imread(filename)

        if image is None:
            print(f"Error loading image: {filename}")
            return None
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        height, width, _ = image.shape

        mid = height // 2

        top = max(0, mid - int(0.34 * height))
        bottom = min(height, mid + int(0.34 * height))

        cropped_image = image[top:bottom, :]

        return cropped_image.astype(np.uint8)

    except Exception as e:
        print(f"Error processing {filename}: {e}")
        return None


def plot_results(
    image: np.ndarray,
    image_path: str,
    image_name: str,
    true_label: str,
    predicted_label_v1: str,
    v1_score: float,
    out_root: str = "./dataset/Todos"
) -> str:
    """
    Gera uma visualização em grid com três painéis: rótulo verdadeiro,
    predição do modelo e mapa de ativação Grad-CAM.

    A função:
    - Pré-processa a imagem para entrada no modelo.
    - Gera o mapa de ativação Grad-CAM.
    - Cria uma figura com três subplots (label real, predição e Grad-CAM).
    - Salva a imagem resultante em um diretório baseado no rótulo verdadeiro
      (grad_positivo ou grad_negativo).

    Args:
        image (np.ndarray): Imagem já carregada em formato array (RGB).
        image_path (str): Caminho da imagem no disco.
        image_name (str): Nome do arquivo da imagem (usado para salvar o resultado).
        true_label (str): Rótulo verdadeiro da imagem.
        predicted_label_v1 (str): Rótulo previsto pelo modelo.
        v1_score (float): Confiança/probabilidade associada à predição.
        out_root (str, optional): Diretório raiz onde as imagens geradas
            serão salvas. Padrão é "./dataset/Todos".

    Returns:
        str: Caminho completo onde a figura gerada foi salva.
    """

    # Pré-processa a imagem e habilita cálculo de gradiente para Grad-CAM
    input_tensor = preprocess_img(image_path)
    input_tensor.requires_grad = True

    # Gera o mapa de ativação Grad-CAM
    cam, _ = generate_cam(gradcam, input_tensor)

    # Cria uma figura com três painéis lado a lado
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f"Image: {image_name}", fontsize=12, fontweight="bold")

    # Painel 1: imagem original com o rótulo verdadeiro
    axes[0].imshow(image)
    axes[0].set_title(f"True Label: {true_label}", fontsize=12)
    axes[0].axis("off")

    # Painel 2: imagem original com a predição do modelo e score
    axes[1].imshow(image)
    axes[1].set_title(f"Prediction: {predicted_label_v1} ({v1_score:.2f})", fontsize=12)
    axes[1].axis("off")

    # Gera a sobreposição do heatmap Grad-CAM na imagem
    heatmap_overlay = overlay_cam(image_path, cam, alpha=0.5, thresh=0.4)

    # Painel 3: visualização do Grad-CAM
    axes[2].imshow(heatmap_overlay)
    axes[2].set_title("Grad-CAM", fontsize=12)
    axes[2].axis("off")

    # Ajusta automaticamente os espaçamentos da figura
    plt.tight_layout()

    # Define o diretório de saída de acordo com o rótulo verdadeiro
    out_dir = "grad_positivo" if true_label == "Positivo" else "grad_negativo"
    out_dir = os.path.join(out_root, out_dir)

    # Garante que o diretório de saída exista
    os.makedirs(out_dir, exist_ok=True)

    # Caminho final do arquivo que será salvo
    out_path = os.path.join(out_dir, image_name)

    # Salva a figura gerada
    fig.savefig(out_path, dpi=200, bbox_inches="tight")

    # Fecha a figura para liberar memória
    plt.close(fig)

    # Retorna o caminho do arquivo salvo
    return out_path


def evaluate_classification(true_labels: list[str], predicted_labels_v1: list[str]) -> None:
    """
    Avalia o desempenho de um modelo de classificação binária (Modelo V1).

    Esta função:
    - Converte rótulos em formato string ("Positivo"/"Negativo") para formato binário (1/0)
    - Calcula e exibe a matriz de confusão
    - Imprime o relatório de classificação contendo precision, recall e F1-score

    Args:
        true_labels (list[str]): Lista com os rótulos verdadeiros (ground-truth).
        predicted_labels_v1 (list[str]): Lista com os rótulos preditos pelo Modelo V1.
    """

    # Converte rótulos string para formato binário (Positivo=1, Negativo=0)
    true_labels_binary = [1 if label == 'Positivo' else 0 for label in true_labels]
    predicted_labels_binary_v1 = [
        1 if label == 'Positivo' else 0 for label in predicted_labels_v1
    ]

    # Força a matriz de confusão a sempre exibir ambas as classes [0,1]
    conf_matrix_v1 = confusion_matrix(
        true_labels_binary,
        predicted_labels_binary_v1,
        labels=[0, 1]
    )

    # Cria o objeto de visualização da matriz de confusão
    disp_v1 = ConfusionMatrixDisplay(
        confusion_matrix=conf_matrix_v1,
        display_labels=['Negativo', 'Positivo']
    )

    # Plota a matriz de confusão
    fig, ax = plt.subplots(figsize=(6, 5))
    disp_v1.plot(cmap=plt.cm.Blues, ax=ax, values_format="d")
    ax.set_title("Confusion Matrix - V1 Model")

    plt.tight_layout()
    plt.show()

    # Imprime métricas detalhadas de classificação
    print("Classification Report - V1 Model:")
    print(
        classification_report(
            true_labels_binary,
            predicted_labels_binary_v1,
            labels=[0, 1],
            target_names=['Negativo', 'Positivo'],
            zero_division=0  # Evita avisos de divisão por zero
        )
    )


def evaluate_directory(
    input_directory: str,
    MODEL_V1,
    out_root: str = None,
    skip_existing: bool = True,
) -> tuple[list[str], list[str]]:
    """
    Avalia todas as imagens de um diretório utilizando o modelo informado.

    A função percorre recursivamente o diretório, realiza predição para cada
    imagem e gera visualizações com Grad-CAM. Também permite retomar
    automaticamente o processamento caso tenha ocorrido uma interrupção
    anterior.

    Funcionalidades principais:
    - Detecta imagens já processadas (presentes em grad_positivo/grad_negativo)
    - Processa apenas arquivos pendentes
    - Salva visualizações com Grad-CAM
    - Armazena rótulos verdadeiros e preditos para posterior avaliação

    Args:
        input_directory (str): Diretório contendo as imagens organizadas
            em subpastas "Positivo" e "Negativo".
        MODEL_V1: Modelo utilizado para realizar as predições.
        out_root (str, optional): Diretório raiz onde serão salvos os resultados
            (grad_positivo e grad_negativo). Se None, utiliza o próprio
            diretório do dataset.
        skip_existing (bool): Se True, ignora imagens já processadas.

    Returns:
        tuple[list[str], list[str]]:
            - Lista de rótulos verdadeiros
            - Lista de rótulos preditos pelo modelo
    """

    true_labels: list[str] = []
    predicted_labels_v1: list[str] = []

    # Se não especificado, usa o próprio diretório do dataset como saída
    if out_root is None:
        out_root = input_directory

    # Lista todos os arquivos do dataset recursivamente
    files = [
        f for f in glob.glob(f"{input_directory}/**/*.*", recursive=True)
        if os.path.isfile(f)
    ]
    total = len(files)
    print(f"Total de imagens encontradas: {total}")

    if total == 0:
        return true_labels, predicted_labels_v1

    # Garante que as pastas de saída existam
    out_pos = os.path.join(out_root, "grad_positivo")
    out_neg = os.path.join(out_root, "grad_negativo")
    os.makedirs(out_pos, exist_ok=True)
    os.makedirs(out_neg, exist_ok=True)

    # Filtra arquivos já processados (permite retomada após falha)
    if skip_existing:
        pending = []
        done = 0
        for f in files:
            name = Path(f).name

            # Se a imagem já existe em qualquer pasta de saída, considera processada
            if os.path.exists(os.path.join(out_pos, name)) or os.path.exists(os.path.join(out_neg, name)):
                done += 1
            else:
                pending.append(f)

        print(f"Já processadas (encontradas em grad_*): {done}")
        print(f"Pendentes para processar agora: {len(pending)}")
    else:
        pending = files
        done = 0
        print("skip_existing=False → vai processar tudo do zero.")

    # Barra de progresso apenas para imagens pendentes
    pbar = tqdm(
        pending,
        total=len(pending),
        desc=f"Processando pendentes (já feitas: {done}/{total})",
        unit="img",
        file=sys.stdout,
        dynamic_ncols=True,
        leave=True
    )

    for k, filename in enumerate(pbar, start=1):
        try:

            # Atualiza informação de progresso exibida na barra
            processed_so_far = done + k
            pbar.set_postfix_str(f"{processed_so_far}/{total} ({(processed_so_far/total)*100:.1f}%) | {Path(filename).name}")

            # Carrega a imagem
            image = read_image(filename)
            if image is None:
                continue

            # Realiza predição com o modelo
            results_v1 = MODEL_V1.predict(filename, verbose=False)
            class_probabilities_v1 = results_v1[0].probs.data.cpu().numpy()

            predicted_label_index_v1 = int(np.argmax(class_probabilities_v1))
            v1_score = float(class_probabilities_v1[predicted_label_index_v1])

            predicted_label_v1 = "Positivo" if predicted_label_index_v1 == 1 else "Negativo"

            # Obtém o rótulo verdadeiro a partir do nome da pasta
            path_obj = Path(filename)
            image_name = path_obj.name
            parent_folder = path_obj.parent.name.lower()

            if parent_folder == "positivo":
                true_label = "Positivo"
            elif parent_folder == "negativo":
                true_label = "Negativo"
            else:
                pbar.write(f"[WARN] Pasta desconhecida para: {filename}")
                continue

            # Armazena rótulos para avaliação posterior
            true_labels.append(true_label)
            predicted_labels_v1.append(predicted_label_v1)

            # Gera e salva a visualização com Grad-CAM
            plot_results(
                image=image,
                image_path=filename,
                image_name=image_name,
                true_label=true_label,
                predicted_label_v1=predicted_label_v1,
                v1_score=v1_score,
                out_root=out_root
            )

        except Exception as e:
            # Caso ocorra erro em uma imagem específica, registra e continua
            pbar.write(f"[ERROR] {Path(filename).name}: {e}")
            continue

    pbar.close()
    return true_labels, predicted_labels_v1


class GradCAM:
    """
    Implementação do algoritmo Grad-CAM (Gradient-weighted Class Activation Mapping).

    O Grad-CAM é uma técnica de interpretabilidade para redes neurais
    convolucionais que permite visualizar quais regiões da imagem
    contribuíram mais para a predição de uma determinada classe.

    A técnica utiliza os gradientes da saída em relação às ativações
    de uma camada convolucional para gerar um mapa de calor
    destacando regiões importantes da imagem.

    Args:
        model (torch.nn.Module): Modelo de rede neural utilizado para inferência.
        target_layer (torch.nn.Module): Camada convolucional usada para gerar
            o mapa de ativação Grad-CAM (normalmente a última camada convolucional).
    """

    def __init__(self, model, target_layer):
        """
        Inicializa o objeto GradCAM.

        Durante a inicialização são registrados hooks na camada alvo
        para capturar automaticamente:

        - As ativações (feature maps) geradas no forward pass
        - Os gradientes calculados no backward pass

        Args:
            model (torch.nn.Module): Modelo utilizado para inferência.
            target_layer (torch.nn.Module): Camada convolucional usada
            para calcular o mapa Grad-CAM.
        """

        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._register_hooks()

    def _register_hooks(self):
        """
        Registra hooks na camada alvo.

        O hook de forward captura os mapas de ativação produzidos
        pela camada durante o forward pass.

        O hook de backward captura os gradientes da pontuação da classe
        alvo em relação aos mapas de ativação dessa camada.
        """

        def forward_hook(_, __, output):
            """Captura os mapas de ativação produzidos pela camada."""
            self.activations = output.detach()

        def backward_hook(_, grad_input, grad_output):
            """Captura os gradientes da classe alvo em relação às ativações."""
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)


def generate_cam(gradcam, input_tensor, class_idx=None):
    """
    Gera um mapa de ativação Grad-CAM para uma imagem de entrada.

    A função realiza:
    1. Forward pass na rede
    2. Backpropagation da classe alvo
    3. Cálculo do Grad-CAM a partir dos gradientes e ativações

    Args:
        gradcam (GradCAM): Objeto GradCAM já inicializado.
        input_tensor (torch.Tensor): Tensor de entrada com formato (B,3,H,W).
        class_idx (int, optional): Índice da classe alvo.
            Se None, utiliza a classe com maior probabilidade (argmax).

    Returns:
        cam (np.ndarray): Mapa Grad-CAM 2D normalizado no intervalo [0,1].
        class_idx (int): Índice da classe utilizada no cálculo.
    """

    # Forward pass no modelo
    output = gradcam.model(input_tensor)
    logits = output[0]

    # Se nenhuma classe foi especificada, usa a classe predita
    if class_idx is None:
        class_idx = logits.argmax(dim=1)

    # Zera gradientes anteriores
    gradcam.model.zero_grad()

    # Seleciona score da classe alvo
    score = logits[:, class_idx]

    # Backpropagation para obter gradientes
    score.backward()

    gradients = gradcam.gradients
    activations = gradcam.activations

    # Calcula pesos globais dos gradientes
    weights = gradients.mean(dim=(2,3), keepdim=True)

    # Combina pesos com ativações
    cam = (weights * activations).sum(dim=1)

    # Aplica ReLU para manter apenas ativações positivas
    cam = F.relu(cam)

    # Normaliza o mapa para intervalo [0,1]
    cam = cam - cam.min()
    cam = cam / cam.max()

    return cam.squeeze().cpu().numpy(), class_idx


def preprocess_img(img_path, img_size=640):
    """
    Realiza o pré-processamento da imagem antes da inferência no modelo.

    Etapas realizadas:
    - Carregamento da imagem
    - Conversão BGR → RGB
    - Recorte vertical central (removendo regiões superiores e inferiores)
    - Redimensionamento para o tamanho esperado pelo modelo
    - Normalização para intervalo [0,1]
    - Conversão para tensor PyTorch

    Args:
        img_path (str): Caminho da imagem.
        img_size (int): Tamanho final da imagem para entrada no modelo.

    Returns:
        torch.Tensor: Tensor da imagem pré-processada com formato (1,3,H,W).
    """

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Obtém dimensões da imagem
    height, width, _ = img.shape

    # Calcula região central para recorte
    mid = height // 2
    top = max(0, mid - int(0.34 * height))
    bottom = min(height, mid + int(0.34 * height))

    # Aplica recorte vertical central
    cropped_image = img[top:bottom, :]

    # Redimensiona para tamanho esperado pela rede
    cropped_image = cv2.resize(cropped_image, (img_size, img_size))

    # Normaliza valores de pixel
    cropped_image = cropped_image.astype(np.float32) / 255.0

    # Converte formato (H,W,C) → (C,H,W)
    cropped_image = np.transpose(cropped_image, (2,0,1))

    # Converte para tensor PyTorch e adiciona dimensão de batch
    image_tensor = torch.tensor(cropped_image).unsqueeze(0)

    return image_tensor


def overlay_cam(
    img_path: str,
    cam: np.ndarray,
    thresh: float = 0.4,
    alpha: float = 0.5,
    colormap: int = cv2.COLORMAP_JET,
    crop_center: bool = True
) -> np.ndarray:
    """
    Carrega uma imagem e sobrepõe o mapa Grad-CAM apenas nas regiões
    com maior ativação.

    O Grad-CAM é convertido em um heatmap colorido e misturado com
    a imagem original apenas nas áreas que ultrapassam um determinado
    limiar de ativação.

    Args:
        img_path (str): Caminho da imagem original.
        cam (np.ndarray): Mapa Grad-CAM normalizado no intervalo [0,1].
        thresh (float): Limiar de ativação para aplicar a sobreposição.
        alpha (float): Fator de mistura entre imagem e heatmap.
        colormap (int): Mapa de cores do OpenCV usado no heatmap.
        crop_center (bool): Se True, aplica o mesmo recorte central
            utilizado no pré-processamento.

    Returns:
        np.ndarray: Imagem RGB contendo a sobreposição do Grad-CAM.
    """

    # Carrega a imagem original
    img = cv2.imread(img_path)
    if img is None:
        raise ValueError(f"Could not read image at {img_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Aplica o mesmo recorte central utilizado no pré-processamento
    if crop_center:
        h, w, _ = img.shape
        mid = h // 2
        top = max(0, mid - int(0.34 * h))
        bottom = min(h, mid + int(0.34 * h))
        img = img[top:bottom, :]

    h, w, _ = img.shape

    # Redimensiona o mapa Grad-CAM para o tamanho da imagem
    cam_resized = cv2.resize(cam, (w, h), interpolation=cv2.INTER_LINEAR)

    # Cria máscara de regiões altamente ativadas
    mask = cam_resized > thresh

    # Gera heatmap colorido
    heatmap = cv2.applyColorMap(
        np.uint8(255 * cam_resized),
        colormap
    )

    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    # Converte imagem e heatmap para float para mistura
    img_float = img.astype(np.float32) / 255.0
    heatmap_float = heatmap.astype(np.float32) / 255.0

    # Mistura heatmap apenas nas regiões ativadas
    overlay = img_float.copy()
    overlay[mask] = (
        (1 - alpha) * img_float[mask] +
        alpha * heatmap_float[mask]
    )

    # Garante que os valores estejam no intervalo válido
    overlay = np.clip(overlay, 0, 1)

    return overlay

In [ ]:
# Acessando a rede interna do YOLO
net = MODEL_V1.model

# Última camada convolucional
target_layer = net.model[-2]

# Inicializando GradCAM
gradcam = GradCAM(net, target_layer)

## Processamento das imagens

In [ ]:
true_labels, predicted_labels = evaluate_directory(
    input_directory="./dataset/Todos",
    MODEL_V1=MODEL_V1,
    out_root="./dataset/Todos",     
    skip_existing=True              
)